<a href="https://colab.research.google.com/github/kaleprafulla1981/kaleprafulla1981/blob/main/DSLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Domain-Specific Language Model (DSLM) with Unsloth

## Hands-on Fine-Tuning Workshop using Google Colab

In this notebook, we will build a Domain-Specific Language Model (DSLM) for **technical support diagnosis** by fine-tuning an existing open-weight language model.

The model will receive a natural-language description of a technical problem such as:

> "My laptop started blue screening while using after I updated the graphics driver."

The model will learn to produce a structured technical-support diagnosis containing:

- Problem Category
- Root Cause
- Recent Change
- Evidence
- Reasoning
- Diagnostic Path
- Repair
- Verification
- Next Step if Failed
- Contrastive Note

### Technology Stack

- **Base Model:** Qwen2.5-1.5B-Instruct
- **Training Method:** Supervised Fine-Tuning (SFT)
- **Fine-Tuning Technique:** QLoRA
- **Framework:** Unsloth
- **Trainer:** TRL SFTTrainer
- **Environment:** Google Colab
- **Model Loading:** 4-bit quantization
- **Output:** Structured JSON diagnosis

The objective is not to train a language model from scratch.

Instead, we will adapt an existing open-weight model to perform a specialized technical-support task.

# 1. Learning Objectives

By the end of this notebook, you will understand how to:

1. Understand what a Domain-Specific Language Model is.
2. Select an open-weight model for domain adaptation.
3. Understand SFT, LoRA and QLoRA.
4. Prepare structured domain-specific training data.
5. Validate a dataset before training.
6. Convert structured examples into chat conversations.
7. Apply a model's native chat template.
8. Load a quantized model using Unsloth.
9. Add LoRA adapters to the model.
10. Fine-tune using Supervised Fine-Tuning.
11. Establish a base-model benchmark.
12. Evaluate the domain-adapted model.
13. Verify JSON and output-schema compliance.
14. Test the model on unseen scenarios.
15. Save and reload the trained LoRA adapter.

# 2. What is a Domain-Specific Language Model?

A Domain-Specific Language Model (DSLM) is a language model adapted to perform tasks within a particular domain.

Examples of domains include:

- Technical Support
- Banking
- Telecommunications
- Healthcare
- Cybersecurity
- Legal Services
- Insurance
- Manufacturing

A general-purpose language model understands many topics.

A domain-specific model is adapted toward the terminology, reasoning patterns, output structures and tasks required within a particular domain.

In this workshop our domain is:

## Technical Support Diagnosis

Our model will learn the transformation:

Natural-Language Technical Problem

↓

Structured Technical Diagnosis

# 3. Are We Training a Model From Scratch?

No.

Training a language model from scratch requires:

- Very large datasets
- Large-scale GPU infrastructure
- Significant compute resources
- Long training cycles

Instead, we start with a pretrained open-weight model.

Our workflow is:

General Open-Weight Model

↓

Domain-Specific Dataset

↓

Supervised Fine-Tuning

↓

Domain Adaptation

↓

Domain-Specific Language Model

This is a much more practical method for creating specialized language models.

# 4. Continued Pretraining vs Supervised Fine-Tuning

Two concepts are important when discussing domain adaptation.

## Continued Pretraining (CPT)

Continued Pretraining exposes the model to large amounts of raw domain text.

Examples include:

- Technical manuals
- Knowledge-base articles
- Service documentation
- Troubleshooting documentation
- Domain publications

The objective is normally still next-token prediction.

CPT is useful when the base model does not sufficiently understand the vocabulary, terminology or knowledge distribution of the target domain.

---

## Supervised Fine-Tuning (SFT)

Supervised Fine-Tuning teaches the model how it should perform a particular task.

The dataset contains:

Input → Expected Output

For our dataset:

Technical Problem

↓

Structured Technical Diagnosis

Our dataset already contains curated input/output examples.

Therefore this workshop uses:

## Supervised Fine-Tuning + QLoRA

# 5. What is LoRA?

LoRA stands for:

## Low-Rank Adaptation

A language model contains billions of parameters.

Updating every parameter during fine-tuning requires significant GPU memory.

LoRA provides a more efficient approach.

Instead of updating the complete model, the original model weights remain largely frozen and small trainable matrices are introduced into selected model layers.

Conceptually:

W' = W + ΔW

LoRA approximates the weight update using two low-rank matrices:

ΔW = BA

Only these relatively small adapter parameters need to be trained.

This makes domain adaptation significantly more computationally efficient.

# 6. What is QLoRA?

QLoRA combines:

## Quantization + LoRA

Instead of loading the complete base model at full precision, the base model is loaded using low-bit quantization.

LoRA adapters are then trained on top of the quantized model.

Conceptually:

4-bit Base Model

+

Trainable LoRA Adapters

↓

QLoRA Fine-Tuning

This significantly reduces GPU memory requirements and makes experimentation with multi-billion-parameter models much more accessible on constrained GPU environments.

# 7. Our DSLM Architecture

Our end-to-end training pipeline will be:

Technical-Support Dataset

↓

Dataset Validation

↓

Dataset Cleaning

↓

Structured Instruction / Output Pairs

↓

Chat Conversation Construction

↓

Qwen Chat Template

↓

Token-Length Analysis

↓

Training / Validation Split

↓

Qwen2.5-3B-Instruct

↓

4-bit Model Loading

↓

LoRA Adapters

↓

Supervised Fine-Tuning

↓

Domain Adapter

↓

Technical-Support DSLM

↓

Evaluation

↓

Save and Reload Model

# 8. Understanding the Dataset

Each training record contains three main fields.

## ID

Example:

tsd_00001

The ID uniquely identifies the record.

It is useful for:

- Tracking examples
- Debugging
- Evaluation
- Dataset management

The ID itself does not need to be shown to the model.

---

## Instruction

The instruction contains the user's technical-support problem in natural language.

Example:

"My Dell Latitude blue screens only while gaming. Office work is fine. I updated the graphics driver immediately before the issue started."

This becomes the USER message during training.

---

## Output

The output contains the expected structured technical diagnosis.

The model will learn to generate this structured response.

# 8. Understanding the Dataset

Each training record contains three main fields.

## ID

Example:

tsd_00001

The ID uniquely identifies the record.

It is useful for:

- Tracking examples
- Debugging
- Evaluation
- Dataset management

The ID itself does not need to be shown to the model.

---

## Instruction

The instruction contains the user's technical-support problem in natural language.

Example:

"My Dell Latitude blue screens only while gaming. Office work is fine. I updated the graphics driver immediately before the issue started."

This becomes the USER message during training.

---

## Output

The output contains the expected structured technical diagnosis.

The model will learn to generate this structured response.

# 10. What is the Model Learning?

The model is learning more than technical terminology.

It is learning a diagnostic process:

User Problem

↓

Identify Symptoms

↓

Identify Recent Change

↓

Extract Evidence

↓

Reason About the Most Likely Cause

↓

Construct Diagnostic Path

↓

Recommend Repair

↓

Define Verification

↓

Provide Next Steps

↓

Explain Why Alternative Actions May Be Less Appropriate

↓

Return Structured JSON

Our fine-tuning objective therefore combines:

1. Domain adaptation
2. Diagnostic behaviour
3. Structured output generation

# 11. Important Principle: Evidence vs Assumption

A diagnostic model should not turn an assumption into a fact.

For example, if the user says:

"The laptop started crashing after a graphics driver update."

the model can use this as evidence supporting a driver regression.

However, if the user does not provide GPU temperature information, the model should not claim:

"The GPU is overheating."

Instead, temperature checking can appear as a diagnostic step.

High-quality domain training data should consistently distinguish between:

- Evidence that is already known
- Inferences supported by that evidence
- Diagnostic steps required to confirm uncertain conditions

# 12. Google Colab Runtime

This notebook requires a GPU runtime.

In Google Colab:

Runtime → Change runtime type → GPU

The next cell displays the GPU currently assigned to the notebook.

In [1]:
!nvidia-smi

Tue Nov  3 08:29:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

##Install Libraries

In [2]:
%%capture

!pip install -U \
    unsloth \
    unsloth_zoo \
    transformers \
    datasets \
    trl \
    peft \
    accelerate \
    bitsandbytes \
    sentencepiece \
    protobuf \
    huggingface_hub

# 13. Verify the Training Environment

Before beginning training we verify:

- PyTorch installation
- CUDA availability
- GPU model
- Available GPU memory

If CUDA is not available, verify that the Colab runtime has been configured to use a GPU.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    gpu_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print("GPU:", gpu_name)
    print(f"GPU memory: {gpu_memory:.2f} GB")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
import os
import json
import html
import random
import numpy as np
import torch

from pprint import pprint
from google.colab import files
from datasets import Dataset

from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"

MAX_SEQ_LENGTH = 2048


LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2

GRADIENT_ACCUMULATION_STEPS = 4


LEARNING_RATE = 2e-4
NUM_TRAIN_EPOCHS = 1

VALIDATION_RATIO = 0.10


OUTPUT_DIR = "/content/technical-support-dslm"

ADAPTER_DIR = "/content/technical-support-dslm-final"

# 14. Training Configuration

For the workshop we begin with the following configuration:

### Model

Qwen2.5-1.5B-Instruct

### Maximum Sequence Length

2048 tokens

### LoRA Rank

16

The LoRA rank controls the adaptation capacity of the trainable adapter.

### LoRA Alpha

32

Alpha controls the scaling applied to the LoRA update.

### Training Batch Size

2

### Gradient Accumulation

4 steps

The approximate effective batch size is therefore:

2 × 4 = 8 examples

### Learning Rate

2e-4

### Training Epochs

3

These values are starting points for the workshop rather than universal optimal values.

Model quality should ultimately be determined through validation and held-out testing.

# 15. Upload the Dataset

The next cell uploads the training dataset from your computer into the current Colab session.

The expected dataset is a JSON file containing an array of training records.

In [ ]:
uploaded = files.upload()

DATASET_PATH = list(uploaded.keys())[0]

print("Dataset:", DATASET_PATH)

Saving troubleshooting_dataset_v2_causal.json to troubleshooting_dataset_v2_causal.json
Dataset: troubleshooting_dataset_v2_causal.json


In [ ]:
with open(
    DATASET_PATH,
    "r",
    encoding="utf-8"
) as file:

    raw_data = json.load(file)


assert isinstance(
    raw_data,
    list
), "The top-level JSON structure must be an array."


print("Total records:", len(raw_data))

print("\nFirst record ID:")
print(raw_data[0].get("id"))

print("\nFirst instruction:")
print(raw_data[0].get("instruction"))

print("\nOutput fields:")
print(
    list(
        raw_data[0].get(
            "output",
            {}
        ).keys()
    )
)

Total records: 4230

First record ID:
tsd_00001

First instruction:
Hey, my Dell Latitude blue screens only while gaming, since last week. Office work and browsing are totally fine. I updated my graphics driver right before this began. Let me know what to check.

Output fields:
['problem_category', 'root_cause', 'recent_change', 'evidence', 'reasoning', 'diagnostic_path', 'repair', 'verification', 'next_step_if_failed', 'contrastive_note']


# 16. Validate the Dataset Schema

Training data should be validated before training begins.

Our dataset requires:

### Top-Level Fields

- id
- instruction
- output

### Required Output Fields

- problem_category
- root_cause
- recent_change
- evidence
- reasoning
- diagnostic_path
- repair
- verification
- next_step_if_failed
- contrastive_note

The following checks detect malformed training examples before GPU training begins.

In [ ]:
REQUIRED_TOP_LEVEL_FIELDS = [
    "id",
    "instruction",
    "output"
]


REQUIRED_OUTPUT_FIELDS = [
    "problem_category",
    "root_cause",
    "recent_change",
    "evidence",
    "reasoning",
    "diagnostic_path",
    "repair",
    "verification",
    "next_step_if_failed",
    "contrastive_note"
]


validation_errors = []


for index, record in enumerate(raw_data):

    record_name = record.get(
        "id",
        f"record_{index}"
    )


    for field in REQUIRED_TOP_LEVEL_FIELDS:

        if field not in record:

            validation_errors.append(
                f"{record_name}: missing top-level field '{field}'"
            )


    if "output" not in record:

        continue


    if not isinstance(
        record["output"],
        dict
    ):

        validation_errors.append(
            f"{record_name}: output must be a JSON object"
        )

        continue


    for field in REQUIRED_OUTPUT_FIELDS:

        if field not in record["output"]:

            validation_errors.append(
                f"{record_name}: missing output field '{field}'"
            )


print(
    "Validation errors:",
    len(validation_errors)
)


for error in validation_errors[:20]:

    print("-", error)


assert (
    len(validation_errors) == 0
), "Dataset schema validation failed."

Validation errors: 0


In [ ]:
type_errors = []


STRING_FIELDS = [
    "problem_category",
    "root_cause",
    "recent_change",
    "reasoning",
    "contrastive_note"
]


LIST_FIELDS = [
    "evidence",
    "diagnostic_path",
    "repair",
    "verification",
    "next_step_if_failed"
]


for record in raw_data:

    record_id = record["id"]

    output = record["output"]


    if not isinstance(
        record["instruction"],
        str
    ):

        type_errors.append(
            f"{record_id}: instruction must be a string"
        )


    for field in STRING_FIELDS:

        if not isinstance(
            output[field],
            str
        ):

            type_errors.append(
                f"{record_id}: {field} must be a string"
            )


    for field in LIST_FIELDS:

        if not isinstance(
            output[field],
            list
        ):

            type_errors.append(
                f"{record_id}: {field} must be a list"
            )


print(
    "Type validation errors:",
    len(type_errors)
)


for error in type_errors[:20]:

    print("-", error)


assert (
    len(type_errors) == 0
), "Dataset type validation failed."

Type validation errors: 0


In [ ]:
diagnostic_path_errors = []


for record in raw_data:

    record_id = record["id"]

    diagnostic_path = (
        record["output"]["diagnostic_path"]
    )


    for index, step in enumerate(
        diagnostic_path
    ):

        if not isinstance(
            step,
            dict
        ):

            diagnostic_path_errors.append(
                f"{record_id}: diagnostic_path[{index}] must be an object"
            )

            continue


        for field in [
            "step",
            "action",
            "rationale"
        ]:

            if field not in step:

                diagnostic_path_errors.append(
                    f"{record_id}: diagnostic_path[{index}] "
                    f"missing '{field}'"
                )


print(
    "Diagnostic path errors:",
    len(diagnostic_path_errors)
)


for error in diagnostic_path_errors[:20]:

    print("-", error)


assert (
    len(diagnostic_path_errors) == 0
), "Diagnostic path validation failed."

Diagnostic path errors: 0


# 17. Normalize HTML Entities

Some datasets may contain HTML-encoded characters.

For example:

&amp;gt;

instead of:

>

Before training, we normalize these values so the model learns natural text rather than unnecessary HTML encoding.

This normalization is performed recursively across:

- Strings
- Lists
- Nested JSON objects

In [ ]:
def decode_html(value):

    if isinstance(value, str):

        return html.unescape(value)


    if isinstance(value, list):

        return [
            decode_html(item)
            for item in value
        ]


    if isinstance(value, dict):

        return {
            key: decode_html(item)
            for key, item in value.items()
        }


    return value


raw_data = [
    decode_html(record)
    for record in raw_data
]


print(
    raw_data[0]
    ["output"]
    ["diagnostic_path"]
    [0]
    ["action"]
)

Open Event Viewer > Windows Logs > System and find the BugCheck (Event ID 41 or 1001) entry matching a crash time.


# 18. Remove Exact Duplicate Examples

Duplicate records can cause certain examples to be overrepresented during training.

We therefore remove exact duplicates based on:

- Instruction
- Expected output

The record ID is intentionally excluded from duplicate detection because two records with different IDs may still contain identical training content.

In [ ]:
unique_records = []

seen = set()


for record in raw_data:

    signature = json.dumps(
        {
            "instruction": record["instruction"],
            "output": record["output"]
        },
        sort_keys=True,
        ensure_ascii=False
    )


    if signature not in seen:

        seen.add(signature)

        unique_records.append(record)


print(
    "Original records:",
    len(raw_data)
)

print(
    "Unique records:",
    len(unique_records)
)


raw_data = unique_records

Original records: 4230
Unique records: 4230


In [ ]:
dataset = Dataset.from_list(
    raw_data
)

print(dataset)

print(
    "\nDataset columns:",
    dataset.column_names
)

Dataset({
    features: ['id', 'instruction', 'output'],
    num_rows: 4230
})

Dataset columns: ['id', 'instruction', 'output']


# 19. Defining the DSLM Behaviour

Our model has two primary objectives:

1. Produce a useful technical diagnosis.
2. Follow the required JSON schema.

The system prompt specifies the desired output contract.

The training examples then demonstrate how this behaviour should be applied to real technical-support cases.

The model should:

- Use the evidence available in the user request.
- Avoid inventing unsupported evidence.
- Identify a likely root cause.
- Explain the reasoning.
- Construct ordered diagnostic actions.
- Recommend repairs.
- Define verification criteria.
- Provide fallback actions.
- Return valid JSON.

In [ ]:
SYSTEM_PROMPT = """
You are a technical support diagnostic assistant.

Analyze the technical problem described by the user.

Identify the most likely root cause using the evidence available in the user's description.

Return ONLY one valid JSON object.

The JSON object must contain exactly these top-level fields:

problem_category
root_cause
recent_change
evidence
reasoning
diagnostic_path
repair
verification
next_step_if_failed
contrastive_note

diagnostic_path must be an array of objects.

Each diagnostic_path object must contain:

step
action
rationale

Do not include Markdown code fences.

Do not include explanatory text before or after the JSON object.

Do not invent evidence that was not supplied by the user.

When the evidence is insufficient to confirm a cause, reflect that uncertainty in the diagnosis and diagnostic path.
""".strip()

# 20. Convert Training Records into Conversations

Instruction-tuned language models are trained using conversational structures.

Our original record:

Instruction

+

Structured Output

will therefore become:

SYSTEM

Defines the expected technical-support behaviour.

↓

USER

Contains the user's technical problem.

↓

ASSISTANT

Contains the expected structured JSON diagnosis.

The dataset ID is retained for tracking but is not shown to the model.

In [ ]:
def create_messages(example):

    output_json = json.dumps(
        example["output"],
        ensure_ascii=False,
        separators=(",", ":")
    )


    return {
        "messages": [

            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content": example[
                    "instruction"
                ].strip()
            },

            {
                "role": "assistant",
                "content": output_json
            }
        ]
    }


dataset = dataset.map(
    create_messages
)


pprint(
    dataset[0]["messages"]
)

Map:   0%|          | 0/4230 [00:00<?, ? examples/s]

[{'content': 'You are a technical support diagnostic assistant.\n'
             '\n'
             'Analyze the technical problem described by the user.\n'
             '\n'
             'Identify the most likely root cause using the evidence available '
             "in the user's description.\n"
             '\n'
             'Return ONLY one valid JSON object.\n'
             '\n'
             'The JSON object must contain exactly these top-level fields:\n'
             '\n'
             'problem_category\n'
             'root_cause\n'
             'recent_change\n'
             'evidence\n'
             'reasoning\n'
             'diagnostic_path\n'
             'repair\n'
             'verification\n'
             'next_step_if_failed\n'
             'contrastive_note\n'
             '\n'
             'diagnostic_path must be an array of objects.\n'
             '\n'
             'Each diagnostic_path object must contain:\n'
             '\n'
             'step\n'
             'act

# 21. Load the Base Model

We now load:

Qwen2.5-3B-Instruct

using a 4-bit Unsloth checkpoint.

The reduced-precision model decreases GPU memory requirements while retaining the pretrained model as the foundation for our domain adaptation.

The original model will remain largely frozen.

Trainable LoRA adapters will be added in the next step.

In [ ]:
model, tokenizer = (
    FastLanguageModel.from_pretrained(

        model_name=MODEL_NAME,

        max_seq_length=MAX_SEQ_LENGTH,

        dtype=None,

        load_in_4bit=True,

        full_finetuning=False
    )
)


print(
    "Loaded model:",
    MODEL_NAME
)

==((====))==  Unsloth 2026.10.11: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loaded model: unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit


# 22. Add LoRA Adapters

We now attach LoRA adapters to selected transformer projection layers.

The target modules include attention projections:

- q_proj
- k_proj
- v_proj
- o_proj

and feed-forward projections:

- gate_proj
- up_proj
- down_proj

These are the parameters that will learn our domain-specific behaviour while most of the base-model weights remain frozen.

In [ ]:
model = FastLanguageModel.get_peft_model(

    model,

    r=LORA_RANK,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=SEED,

    use_rslora=False,

    loftq_config=None
)

Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


In [ ]:
trainable_parameters = 0
total_parameters = 0


for parameter in model.parameters():

    total_parameters += (
        parameter.numel()
    )

    if parameter.requires_grad:

        trainable_parameters += (
            parameter.numel()
        )


percentage = (
    100
    * trainable_parameters
    / total_parameters
)


print(
    f"Trainable parameters: "
    f"{trainable_parameters:,}"
)

print(
    f"Total parameters: "
    f"{total_parameters:,}"
)

print(
    f"Trainable percentage: "
    f"{percentage:.4f}%"
)

Trainable parameters: 18,464,768
Total parameters: 907,081,216
Trainable percentage: 2.0356%


# 23. Apply the Model's Chat Template

Different instruction models expect conversations to be represented using different special tokens.

Rather than manually constructing these tokens, we use the tokenizer's native chat template.

This converts:

SYSTEM → USER → ASSISTANT

into the exact text representation expected by the model.

This is the text that will ultimately be tokenized during supervised fine-tuning.

In [ ]:
def format_chat(examples):

    texts = []


    for messages in examples["messages"]:

        formatted_text = (
            tokenizer.apply_chat_template(

                messages,

                tokenize=False,

                add_generation_prompt=False
            )
        )

        texts.append(
            formatted_text
        )


    return {
        "text": texts
    }


dataset = dataset.map(
    format_chat,
    batched=True
)


print(
    dataset[0]["text"]
)

Map:   0%|          | 0/4230 [00:00<?, ? examples/s]

<|im_start|>system
You are a technical support diagnostic assistant.

Analyze the technical problem described by the user.

Identify the most likely root cause using the evidence available in the user's description.

Return ONLY one valid JSON object.

The JSON object must contain exactly these top-level fields:

problem_category
root_cause
recent_change
evidence
reasoning
diagnostic_path
repair
verification
next_step_if_failed
contrastive_note

diagnostic_path must be an array of objects.

Each diagnostic_path object must contain:

step
action
rationale

Do not include Markdown code fences.

Do not include explanatory text before or after the JSON object.

Do not invent evidence that was not supplied by the user.

When the evidence is insufficient to confirm a cause, reflect that uncertainty in the diagnosis and diagnostic path.<|im_end|>
<|im_start|>user
Hey, my Dell Latitude blue screens only while gaming, since last week. Office work and browsing are totally fine. I updated my grap

# 24. Analyze Sequence Lengths

Models process text as tokens rather than characters or words.

Long technical-support examples can consume large numbers of tokens because they may contain:

- Detailed reasoning
- Multiple evidence points
- Large diagnostic paths
- Repair instructions
- Verification steps

Before training, we inspect the token-length distribution.

This helps detect examples that exceed our configured maximum sequence length.

In [ ]:
def calculate_token_length(example):

    encoded = tokenizer(

        example["text"],

        add_special_tokens=False,

        truncation=False
    )


    return {
        "token_length":
        len(encoded["input_ids"])
    }


dataset = dataset.map(
    calculate_token_length
)

Map:   0%|          | 0/4230 [00:00<?, ? examples/s]

In [ ]:
lengths = dataset[
    "token_length"
]


print(
    "Minimum:",
    min(lengths)
)

print(
    "Median:",
    int(np.median(lengths))
)

print(
    "90th percentile:",
    int(np.percentile(lengths, 90))
)

print(
    "95th percentile:",
    int(np.percentile(lengths, 95))
)

print(
    "Maximum:",
    max(lengths)
)


over_limit = sum(

    length > MAX_SEQ_LENGTH

    for length in lengths
)


print(
    f"\nExamples above "
    f"{MAX_SEQ_LENGTH} tokens:",
    over_limit
)

Minimum: 571
Median: 654
90th percentile: 705
95th percentile: 713
Maximum: 818

Examples above 2048 tokens: 0


In [ ]:
long_examples = [

    (
        dataset[index]["id"],
        dataset[index]["token_length"]
    )

    for index
    in range(len(dataset))

    if dataset[index]["token_length"]
    > MAX_SEQ_LENGTH
]


long_examples = sorted(

    long_examples,

    key=lambda item: item[1],

    reverse=True
)


for record_id, length in long_examples[:20]:

    print(
        record_id,
        length
    )

# 25. Handle Oversized Examples

Blindly truncating a training example can be problematic.

For example, truncation could remove:

- Verification instructions
- Fallback actions
- Contrastive reasoning
- Part of the JSON structure

This can teach the model incomplete output structures.

For this workshop we remove examples exceeding the configured maximum sequence length.

In a production dataset, long samples should be manually reviewed before removal.

Alternative approaches include:

- Removing unnecessary repetition
- Simplifying verbose diagnostic paths
- Increasing the supported sequence length when hardware permits

In [ ]:
original_count = len(
    dataset
)

dataset = dataset.filter(
    lambda example:
    example["token_length"]
    <= MAX_SEQ_LENGTH
)

print(
    "Before length filtering:",
    original_count
)

print(
    "After length filtering:",
    len(dataset)
)

Filter:   0%|          | 0/4230 [00:00<?, ? examples/s]

Before length filtering: 4230
After length filtering: 4230


# 26. Create Training and Validation Splits

The model should not be evaluated solely using examples on which it was trained.

We divide the available dataset into:

## Training Set

Used to update the LoRA parameters.

## Validation Set

Used to monitor the model during training.

For a rigorous model evaluation, a separate held-out test dataset should also be maintained.

Care should be taken to avoid highly similar scenarios appearing across training and evaluation sets because this can produce misleadingly strong results.

In [ ]:
assert len(dataset) >= 2, (
    "At least two usable records are required "
    "after filtering."
)

validation_count = max(
    1,
    int(
        len(dataset)
        * VALIDATION_RATIO
    )
)

validation_count = min(
    validation_count,
    len(dataset) - 1
)

splits = (
    dataset.train_test_split(
        test_size=validation_count,
        seed=SEED,
        shuffle=True
    )
)


train_dataset = (
    splits["train"]
)

eval_dataset = (
    splits["test"]
)

print(
    "Training records:",
    len(train_dataset)
)

print(
    "Validation records:",
    len(eval_dataset)
)

Training records: 3807
Validation records: 423


# 27. Establish a Base-Model Baseline

We should measure the original model before fine-tuning it.

A held-out technical-support scenario will be sent to the base model.

The response will be saved.

After fine-tuning, the exact same scenario will be sent to the DSLM.

This gives us:

Base Model

vs.

Domain-Specific Fine-Tuned Model

A meaningful DSLM experiment should demonstrate improvement rather than simply demonstrate that training completed successfully.

In [ ]:
def generate_diagnosis(
    instruction,
    model,
    tokenizer,
    max_new_tokens=1200
):

    FastLanguageModel.for_inference(
        model
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": instruction.strip()
        }
    ]

    prompt = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    input_length = (
        inputs[
            "input_ids"
        ].shape[1]
    )

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True,
            pad_token_id=(
                tokenizer.eos_token_id
            )
        )

    generated_tokens = (
        output[0][input_length:]
    )


    response = (
        tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )
    )

    return response.strip()

In [ ]:
BASELINE_TEST = """
My laptop randomly disconnects from Wi-Fi after waking from sleep.
The connection works normally again after I disable and re-enable the wireless adapter.
The issue started immediately after I installed a new wireless network driver.
All other devices connected to the same Wi-Fi network are working normally.
"""

In [ ]:
baseline_response = (
    generate_diagnosis(
        BASELINE_TEST,
        model,
        tokenizer
    )
)

print(
    "BASE MODEL RESPONSE"
)
print(
    "=" * 80
)
print(
    baseline_response
)

Both `max_new_tokens` (=1200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


BASE MODEL RESPONSE
```json
{
  "problem_category": "Network Connectivity",
  "root_cause": "Driver Update",
  "recent_change": "Installed a new wireless network driver",
  "evidence": [
    "Laptop randomly disconnects from Wi-Fi after waking from sleep",
    "Connection works normally after disabling and re-enabling the wireless adapter"
  ],
  "reasoning": "The issue started immediately after installing a new driver, suggesting that the driver update caused the problem.",
  "diagnostic_path": [
    {
      "step": 1,
      "action": "Check for any recent updates or changes to the wireless network drivers",
      "rationale": "Review the system logs or device manager to identify if there were any recent updates or changes to the wireless network drivers."
    },
    {
      "step": 2,
      "action": "Disable and re-enable the wireless adapter",
      "rationale": "Try disabling and re-enabling the wireless adapter to see if the issue persists."
    }
  ],
  "repair": "Reinstall the 

# 28. Validate Generated JSON

Our DSLM does not only need to generate a technically useful response.

The response also needs to respect the expected structured format.

One of our first automated evaluations is therefore:

Can the response be parsed successfully as JSON?

If JSON parsing fails, a downstream application may not be able to consume the response reliably.

In [ ]:
def parse_json_response(
    response
):
    try:
        parsed = json.loads(
            response
        )
        return {
            "valid_json": True,
            "data": parsed,
            "error": None
        }

    except Exception as error:
        return {
            "valid_json": False,
            "data": None,
            "error": str(error)
        }

In [ ]:
baseline_json_check = (
    parse_json_response(
        baseline_response
    )
)

print(
    "Base model produced valid JSON:",
    baseline_json_check[
        "valid_json"
    ]
)

if not baseline_json_check[
    "valid_json"
]:
    print(
        "JSON parsing error:"
    )
    print(
        baseline_json_check[
            "error"
        ]
    )

Base model produced valid JSON: False
JSON parsing error:
Expecting value: line 1 column 1 (char 0)


# 29. Configure Supervised Fine-Tuning

We are now ready to train the LoRA adapters.

During training:

1. The technical-support scenario is given to the model.
2. The expected JSON diagnosis acts as the target.
3. The model calculates prediction error.
4. Gradients are computed.
5. LoRA adapter parameters are updated.
6. The pretrained base model remains largely frozen.
7. The process repeats over the training dataset.

Validation is periodically performed using examples that are not used for parameter updates.

In [ ]:
use_bf16 = (

    torch.cuda.is_available()
    and
    torch.cuda.is_bf16_supported()
)


print(
    "Using BF16:",
    use_bf16
)


training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=(
        NUM_TRAIN_EPOCHS
    ),
    per_device_train_batch_size=(
        TRAIN_BATCH_SIZE
    ),
    per_device_eval_batch_size=(
        EVAL_BATCH_SIZE
    ),
    gradient_accumulation_steps=(
        GRADIENT_ACCUMULATION_STEPS
    ),
    learning_rate=(
        LEARNING_RATE
    ),
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=25,
    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=not use_bf16,
    bf16=use_bf16,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    packing=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED
)

Using BF16: False
Unsloth: transformers renamed `warmup_ratio` to `warmup_steps`. Forwarding your value to `warmup_steps` - update your code when convenient. If you also passed `warmup_steps` as 0.1, that is its default here and cannot be distinguished from leaving it unset, so `warmup_ratio` was used; drop `warmup_ratio` to keep it.


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=(
        train_dataset
    ),
    eval_dataset=(
        eval_dataset
    ),
    args=training_args
)


print(
    "Trainer initialized successfully."
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3807 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/423 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Trainer initialized successfully.


# 30. Train the DSLM

The next cell starts supervised fine-tuning.

During training, monitor:

- Training loss
- Evaluation loss
- Training progress
- GPU memory

A decreasing training loss means the model is becoming better at predicting the expected training outputs.

However, training loss alone does not prove that the DSLM has become a better technical-support model.

We will evaluate its behaviour separately after training.

In [ ]:
training_result = (
    trainer.train()
)
print(
    training_result.metrics
)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,807 | Num Epochs = 1 | Total steps = 476
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Step,Training Loss,Validation Loss
25,1.502025,1.390617
50,0.861067,0.795243
75,0.351817,0.275403
100,0.115896,0.107208
125,0.073452,0.061842
150,0.043837,0.042382
175,0.037497,0.035349
200,0.032179,0.030364
225,0.028617,0.027689
250,0.026793,0.026096


Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-125/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-175/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-225/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/technical-support-dslm/checkpoint-275/tokenizer_config.json.


KeyboardInterrupt: 

# 31. Evaluate Validation Loss

Validation data is not used to update the LoRA parameters.

Validation metrics therefore provide a better indication of generalization than training loss alone.

However, validation loss is still only one evaluation signal.

For a technical-support DSLM we must also evaluate:

- Diagnostic accuracy
- Evidence grounding
- JSON compliance
- Schema compliance
- Diagnostic ordering
- Repair quality
- Verification quality

In [ ]:
evaluation_results = (
    trainer.evaluate()
)

print(
    "Validation results:"
)

for key, value in (
    evaluation_results.items()
):
    print(
        f"{key}: {value}"
    )

# 32. Test the Fine-Tuned DSLM

We now submit the exact same technical-support scenario used for our baseline.

This allows us to compare:

Base Qwen Model

vs.

Fine-Tuned Technical-Support DSLM

We are looking for improvement in both:

## Semantic Quality

- Problem classification
- Root-cause reasoning
- Evidence usage
- Diagnostic steps
- Repair recommendation
- Verification
- Fallback actions

## Structural Quality

- Valid JSON
- Correct fields
- Correct diagnostic_path structure

In [ ]:
fine_tuned_response = (
    generate_diagnosis(
        BASELINE_TEST,
        model,
        tokenizer
    )
)

print("BASE MODEL")
print("=" * 80)
print(baseline_response)

print("\n\nFINE-TUNED DSLM")
print("=" * 80)
print(fine_tuned_response)

# 33. Schema Evaluation

A response may be valid JSON but still violate the expected schema.

For example, the model could:

- Omit root_cause
- Rename evidence
- Add unexpected top-level fields
- Generate diagnostic_path as a string
- Omit rationale from diagnostic steps

Therefore we perform separate schema validation.

In [ ]:
EXPECTED_FIELDS = {
    "problem_category",
    "root_cause",
    "recent_change",
    "evidence",
    "reasoning",
    "diagnostic_path",
    "repair",
    "verification",
    "next_step_if_failed",
    "contrastive_note"
}


def evaluate_json_structure(
    response
):
    result = {
        "valid_json": False,
        "all_fields_present": False,
        "missing_fields": [],
        "extra_fields": [],
        "diagnostic_path_valid": False
    }


    try:
        parsed = json.loads(
            response
        )
        result[
            "valid_json"
        ] = True


    except Exception as error:
        result[
            "error"
        ] = str(error)
        return result


    if not isinstance(
        parsed,
        dict
    ):
        result[
            "error"
        ] = (
            "Top-level JSON must be an object."
        )
        return result

    actual_fields = set(
        parsed.keys()
    )

    missing_fields = (
        EXPECTED_FIELDS
        - actual_fields
    )

    extra_fields = (
        actual_fields
        - EXPECTED_FIELDS
    )

    result[
        "missing_fields"
    ] = sorted(
        list(missing_fields)
    )

    result[
        "extra_fields"
    ] = sorted(
        list(extra_fields)
    )

    result[
        "all_fields_present"
    ] = (

        len(missing_fields) == 0
        and
        len(extra_fields) == 0
    )

    diagnostic_path = (
        parsed.get(
            "diagnostic_path"
        )
    )

    if isinstance(
        diagnostic_path,
        list
    ):
        path_valid = all(
            isinstance(
                step,
                dict
            )
            and
            {
                "step",
                "action",
                "rationale"
            }.issubset(
                step.keys()
            )

            for step
            in diagnostic_path
        )


        result[
            "diagnostic_path_valid"
        ] = path_valid


    return result

In [ ]:
base_structure = (
    evaluate_json_structure(
        baseline_response
    )
)

dslm_structure = (
    evaluate_json_structure(
        fine_tuned_response
    )
)
print(
    "BASE MODEL STRUCTURE"
)
print(
    json.dumps(
        base_structure,
        indent=2
    )
)
print(
    "\nFINE-TUNED DSLM STRUCTURE"
)
print(
    json.dumps(
        dslm_structure,
        indent=2
    )
)

# 34. Evaluating the DSLM

A useful technical-support DSLM should be evaluated across several dimensions.

## 1. JSON Validity

Can the response be parsed successfully?

## 2. Schema Compliance

Are the required fields present with the expected structure?

## 3. Problem Classification

Did the model identify the appropriate problem category?

## 4. Root Cause

Is the proposed cause supported by available evidence?

## 5. Evidence Grounding

Did the model use information supplied by the user rather than inventing evidence?

## 6. Reasoning

Does the reasoning logically connect symptoms and recent changes to the suspected cause?

## 7. Diagnostic Path

Are troubleshooting actions ordered logically?

## 8. Repair

Does the repair address the diagnosed problem?

## 9. Verification

Does the model explain how to determine whether the repair succeeded?

## 10. Failure Handling

Does the model provide sensible next steps if the initial repair fails?

## 11. Contrastive Reasoning

Does the model explain why another apparently plausible action may be inappropriate?

# 35. Test Generalization with an Unseen Scenario

The final objective is not memorization.

A useful DSLM should apply learned diagnostic patterns to scenarios that were not present in its training data.

We therefore test the model using a completely new technical-support problem.

The test scenario should not be copied from the training dataset.

In [ ]:
UNSEEN_TEST = """
My Windows laptop suddenly became very slow after I installed a new application yesterday.

Task Manager shows that the application keeps using nearly all available CPU even when I am not actively using it.

Memory usage and disk usage look normal.

The laptop becomes responsive again immediately after I close the application from Task Manager.

What should I check?
"""

In [ ]:
unseen_response = (
    generate_diagnosis(
        UNSEEN_TEST,
        model,
        tokenizer
    )
)

print(unseen_response)

In [ ]:
unseen_structure = (
    evaluate_json_structure(
        unseen_response
    )
)
print(
    json.dumps(
        unseen_structure,
        indent=2
    )
)

# 36. Memorization vs Generalization

The model should not simply memorize:

Specific Error → Specific Answer

Instead, it should learn reusable diagnostic patterns.

For example:

Symptom

+

Recent Change

+

Supporting Evidence

↓

Likely Cause

↓

Diagnostic Path

↓

Repair

↓

Verification

This ability to transfer learned behaviour to new problems is called generalization.

Testing unseen scenarios is therefore essential when evaluating a DSLM.

# 37. Save the Domain Adapter

QLoRA does not require us to save another complete copy of the base model for every domain experiment.

We can save the trained LoRA adapter.

Conceptually:

Qwen2.5-3B-Instruct

+

Technical-Support LoRA Adapter

=

Technical-Support DSLM

The adapter represents the learned domain specialization.

In [ ]:
model.save_pretrained(ADAPTER_DIR)

tokenizer.save_pretrained(ADAPTER_DIR)


print("Adapter saved to:")

print(ADAPTER_DIR)

# Optional: Save a Merged Model

The LoRA adapter is generally the smallest training artifact to retain.

For deployment scenarios, the adapter can also be merged with the base model.

This creates a standalone model containing the learned adaptation rather than requiring the LoRA adapter to be attached separately.

The merged model occupies significantly more storage than the adapter alone.

In [ ]:
MERGED_MODEL_DIR = (
    "/content/"
    "technical-support-dslm-merged"
)

model.save_pretrained_merged(
    MERGED_MODEL_DIR,
    tokenizer,
    save_method="merged_16bit"
)
print("Merged model saved to:")

print(MERGED_MODEL_DIR)

# 41. Optional: Export to GGUF

For some local inference workflows, the model can also be exported to GGUF.

This is useful when the trained model will later be used with compatible local inference runtimes.

For this workshop, exporting the model is optional.

The main training artifact remains the LoRA adapter.

In [ ]:
GGUF_DIR = (
    "/content/"
    "technical-support-dslm-gguf"
)
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m"
)
print("GGUF export saved to:")

print(GGUF_DIR)